In [ ]:

import pickle
from collections import defaultdict
from google.colab import drive

# === 0. Mount Google Drive ===
drive.mount('/content/drive')
topk_paths = [
    "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_tiny_imageNet_topk.pkl",
    "/content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task8CS_tiny_imageNet_topk.pkl",
]
neigh_paths = [
    "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_tiny_imageNet_neighbors.pkl",
    "/content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task8CS_tiny_imageNet_neighbors.pkl",
]

out_topk_path  = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_topk.pkl"
out_neigh_path = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_neighbors.pkl"

def load_list(p):
    with open(p, "rb") as f:
        return pickle.load(f)

# 1) Union of Top-K (sum CS values if present)
topk_union = defaultdict(lambda: {"name": None, "index": None, "cs": 0.0})
for p in topk_paths:
    for e in load_list(p):
        key = (e["name"], int(e["index"]))
        if topk_union[key]["name"] is None:
            topk_union[key]["name"]  = e["name"]
            topk_union[key]["index"] = int(e["index"])
        topk_union[key]["cs"] += float(e.get("cs", 0.0))

topk_merged = list(topk_union.values())
topk_keys   = {(d["name"], d["index"]) for d in topk_merged}

# 2) Union of Neighbors (sum) while excluding Top-K to prevent overlap
neigh_union = defaultdict(lambda: {"name": None, "index": None, "position": (), "cs": 0.0})
for p in neigh_paths:
    for it in load_list(p):
        key = (it["name"], int(it["index"]))
        if key in topk_keys:
            continue  # Prevent overlap: a parameter cannot be both a neighbor and frozen
        if neigh_union[key]["name"] is None:
            neigh_union[key]["name"]  = it["name"]
            neigh_union[key]["index"] = int(it["index"])
            if "position" in it:
                neigh_union[key]["position"] = tuple(it["position"])
        neigh_union[key]["cs"] += float(it.get("cs", 0.0))

neighbors_merged = list(neigh_union.values())
# 3) Final check (must be 0)
inter = sum(1 for n in neighbors_merged if (n["name"], n["index"]) in topk_keys)
print("Intersection(Neighbors, TopK) =", inter)

# 4) Save as is (without truncation/percentage)
with open(out_topk_path,  "wb") as f: pickle.dump(topk_merged, f)
with open(out_neigh_path, "wb") as f: pickle.dump(neighbors_merged, f)

print(f"TopK merged: {len(topk_merged)} | Neighbors merged: {len(neighbors_merged)}")


Mounted at /content/drive
Intersection(Neighbors, TopK) = 0
TopK merged: 2400154 | Neighbors merged: 825233
